# NB-3 - Context Lifecycle Management

**Context Engineering with Tools & MCP on Databricks**

This notebook addresses what happens when context is already filling up:
1. **Pruning raw tool outputs with Provence** - sentence-level semantic compression of tool results
2. **Packaging Agent Skills** - refactoring an overloaded system prompt into on-demand Skills

### Learning objectives
- Given an agent with a deep message history approaching context capacity, use Provence to
  semantically prune tool outputs to only the sentences relevant to the current task
- Given a Databricks agent whose system prompt is overloaded with rarely-invoked capability
  instructions, identify which capabilities are candidates for packaging as Agent Skills and
  select the loading strategy

### Prerequisites
- A `.env` file with `DATABRICKS_HOST` and `DATABRICKS_TOKEN`
- A model serving endpoint named `claude_45`
- `pip install databricks-sdk databricks-langchain langgraph mlflow tiktoken transformers torch python-dotenv`


## 0 - Install dependencies

In [ ]:
# Run once in your terminal (or uncomment and run this cell)
# pip install databricks-sdk databricks-langchain langgraph mlflow tiktoken transformers torch python-dotenv


## 1 - Imports and config

In [ ]:
import json
import os
import time
import uuid
import mlflow
import tiktoken
from dotenv import load_dotenv
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langchain_core.messages import (
    AnyMessage, AIMessage, HumanMessage, SystemMessage, ToolMessage
)

load_dotenv()

# TODO: fill in your catalog, schema, and SQL warehouse id
CATALOG      = "demo"
SCHEMA       = "context"
WAREHOUSE_ID = "your_warehouse_id"   # TODO

# Provence: skip pruning for tool results smaller than this (no overhead for tiny payloads)
SMALL_RESULT_THRESHOLD = 100          # tokens
MODEL_CONTEXT_WINDOW   = 200_000      # Claude claude_45

w   = WorkspaceClient()
llm = ChatDatabricks(endpoint="claude_45")
enc = tiktoken.get_encoding("cl100k_base")

mlflow.set_experiment("/Users/me/nb3_context_lifecycle")
print(f"Workspace host: {w.config.host}")


## 2 - Helper: run SQL via the Statement Execution API

In [ ]:
def sql(statement: str, warehouse_id: str = WAREHOUSE_ID) -> list[dict]:
    """Execute a SQL statement on a Databricks SQL warehouse and return the results."""
    response = w.statement_execution.execute_statement(
        warehouse_id=warehouse_id,
        statement=statement,
        wait_timeout="30s",
    )
    while response.status.state not in (
        StatementState.SUCCEEDED, StatementState.FAILED,
        StatementState.CANCELED, StatementState.CLOSED,
    ):
        time.sleep(1)
        response = w.statement_execution.get_statement(response.statement_id)

    if response.status.state != StatementState.SUCCEEDED:
        raise RuntimeError(f"SQL failed [{response.status.state}]: {response.status.error}")

    if response.result is None or response.result.data_array is None:
        return []

    cols = [c.name for c in response.manifest.schema.columns]
    return [dict(zip(cols, row)) for row in response.result.data_array]

print("sql() helper ready")


---
# Part 1 - Pruning raw tool outputs with Provence

## 3 - Simulate a multi-turn agent with a filling context window

We build a realistic 6-turn message history with tool results of varying size:
some are small JSON objects, others are large table scans.


In [ ]:
def make_large_payload(label: str, rows: int = 200) -> str:
    """Simulate a bulky tool result such as a table scan or large retrieval chunk."""
    return json.dumps({
        "source":    label,
        "row_count": rows,
        "data":      [{"id": i, "value": f"row_{i}_{label}"} for i in range(rows)]
    })


deep_history = [
    # Turn 1 - large product catalog
    HumanMessage(content="Get me the full product catalog"),
    AIMessage(content="", tool_calls=[{"id": "t1", "name": "get_product_catalog", "args": {}}]),
    ToolMessage(content=make_large_payload("product_catalog", rows=300), tool_call_id="t1"),
    AIMessage(content="The catalog contains 300 items."),

    # Turn 2 - large revenue report
    HumanMessage(content="Now get Q1 revenue by region"),
    AIMessage(content="", tool_calls=[{"id": "t2", "name": "run_revenue_report", "args": {"period": "Q1"}}]),
    ToolMessage(content=make_large_payload("revenue_report_q1", rows=50), tool_call_id="t2"),
    AIMessage(content="Q1 revenue report retrieved. I have the regional breakdown."),

    # Turn 3 - small account lookup
    HumanMessage(content="What plan is customer C-1042 on?"),
    AIMessage(content="", tool_calls=[{"id": "t3", "name": "lookup_account_details_v2", "args": {"account_id": "C-1042"}}]),
    ToolMessage(content=json.dumps({"account_id": "C-1042", "plan": "enterprise", "billing_email": "c1042@corp.com"}), tool_call_id="t3"),
    AIMessage(content="Customer C-1042 is on the enterprise plan."),

    # Turn 4 - large support docs retrieval
    HumanMessage(content="Search for support docs about login errors"),
    AIMessage(content="", tool_calls=[{"id": "t4", "name": "ai_search", "args": {"query": "login errors"}}]),
    ToolMessage(content=make_large_payload("support_docs_login", rows=100), tool_call_id="t4"),
    AIMessage(content="Found 100 support documents related to login errors."),

    # Turn 5 - small discount calc
    HumanMessage(content="What is the discount for customer C-1042 on a 500-unit order?"),
    AIMessage(content="", tool_calls=[{"id": "t5", "name": "calculate_discount", "args": {"account_id": "C-1042", "volume": 500}}]),
    ToolMessage(content=json.dumps({"discount_pct": 12.5}), tool_call_id="t5"),
    AIMessage(content="The discount for C-1042 on 500 units is 12.5%."),

    # Turn 6 - current turn
    HumanMessage(content="Now create a quote for customer C-1042 for 500 units of SKU-007"),
]


def count_message_tokens(messages: list) -> int:
    return sum(len(enc.encode(m.content or "")) for m in messages if hasattr(m, "content"))


total_tokens = count_message_tokens(deep_history)
pct_used     = 100 * total_tokens / MODEL_CONTEXT_WINDOW

print(f"Message history : {len(deep_history)} messages")
print(f"Total tokens    : ~{total_tokens:,}")
print(f"Context usage   : {pct_used:.1f}% of {MODEL_CONTEXT_WINDOW:,}-token window")
if pct_used >= 75:
    print("WARNING: Context above 75% -- pruning recommended before next turn")


## 4 - Sentence-level pruning with Provence

Instead of heuristic rules (size, age), we use **Provence** (`naver/provence-reranker-debertav3-v1`) --
a 430M-parameter DeBERTa-based model from Naver Labs Europe trained for context pruning in RAG pipelines.

**How it works:**
- Input: a *question* (the current agent task) + a *context passage* (the tool result)
- Output: a pruned passage keeping only sentences relevant to the question, plus a relevance score
- Architecture: sequence labelling -- each sentence gets a binary keep/discard label in a single forward pass
- No threshold tuning required: Provence automatically detects how many sentences to retain

**Why this is better than heuristics for tool outputs:**
- Heuristics evict entire ToolMessages based on size/age -- blunt and lossy
- Provence is semantic: it keeps the sentences that answer the current query even inside a large result
- A 300-row product catalog can be reduced to the 2 sentences relevant to the current quote

**License note:** CC BY-NC 4.0 (non-commercial).
For commercial use see `naver/xprovence` or `zilliz/semantic-highlight-bilingual-v1` (MIT).

### Load Provence


In [ ]:
from transformers import AutoModel

# Runs on CPU for moderate-sized tool results; faster on a GPU-enabled cluster
provence = AutoModel.from_pretrained(
    "naver/provence-reranker-debertav3-v1",
    trust_remote_code=True,
)
print("Provence loaded")


## 5 - Define the Provence pruning functions

In [ ]:
def prune_tool_result_with_provence(
    query: str,
    tool_result: str,
    min_tokens: int = SMALL_RESULT_THRESHOLD,
) -> dict:
    """
    Use Provence to prune a single tool result to only the sentences
    relevant to the current query.

    Args:
        query       : The current user question / agent task.
        tool_result : Raw tool output as a string (JSON or plain text).
        min_tokens  : Results smaller than this are returned as-is.

    Returns:
        dict with keys:
            pruned_content   : Pruned text (or original if too small)
            reranking_score  : Provence relevance score for the whole passage
            tokens_before    : Token count before pruning
            tokens_after     : Token count after pruning
            was_pruned       : Whether Provence modified the content
    """
    tokens_before = len(enc.encode(tool_result))

    # Skip Provence for small results -- not worth the overhead
    if tokens_before < min_tokens:
        return {
            "pruned_content":  tool_result,
            "reranking_score": None,
            "tokens_before":   tokens_before,
            "tokens_after":    tokens_before,
            "was_pruned":      False,
        }

    # Provence expects plain text -- convert JSON to readable sentences
    try:
        parsed = json.loads(tool_result)
        if isinstance(parsed, dict):
            context_text = "\n".join(f"{k}: {v}" for k, v in parsed.items() if k != "data")
            if "data" in parsed:
                n = len(parsed["data"])
                context_text += f"\ndata: contains {n} rows from source '{parsed.get('source', 'unknown')}'"
        elif isinstance(parsed, list):
            context_text = "\n".join(str(item) for item in parsed[:20])
        else:
            context_text = str(parsed)
    except (json.JSONDecodeError, TypeError):
        context_text = tool_result

    result          = provence.process(query, context_text)
    pruned_content  = result.get("pruned_context", context_text)
    reranking_score = result.get("reranking_score", None)
    tokens_after    = len(enc.encode(pruned_content))

    return {
        "pruned_content":  pruned_content,
        "reranking_score": reranking_score,
        "tokens_before":   tokens_before,
        "tokens_after":    tokens_after,
        "was_pruned":      pruned_content != context_text,
    }


def prune_history_with_provence(
    messages: list,
    current_query: str,
) -> tuple[list, dict]:
    """
    Apply Provence pruning to every ToolMessage in a message history.

    Args:
        messages      : Full message list.
        current_query : The current user task -- used as the Provence question.

    Returns:
        (pruned_messages, stats_dict)
    """
    pruned = list(messages)
    stats  = {"tools_processed": 0, "tools_pruned": 0,
               "tokens_before": 0,  "tokens_after": 0}

    tool_names = {}
    for m in messages:
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                tool_names[tc["id"]] = tc["name"]

    for i, msg in enumerate(messages):
        if not isinstance(msg, ToolMessage):
            continue

        tool_name = tool_names.get(msg.tool_call_id, "unknown")
        result    = prune_tool_result_with_provence(current_query, msg.content or "")

        stats["tools_processed"] += 1
        stats["tokens_before"]   += result["tokens_before"]
        stats["tokens_after"]    += result["tokens_after"]

        if result["was_pruned"]:
            stats["tools_pruned"] += 1
            pruned[i] = ToolMessage(
                content=result["pruned_content"],
                tool_call_id=msg.tool_call_id,
            )
            print(f"  [{tool_name}] {result['tokens_before']:,} -> {result['tokens_after']:,} tokens "
                  f"(score: {result['reranking_score']:.3f})")
        else:
            reason = "too small" if result["tokens_before"] < SMALL_RESULT_THRESHOLD else "already dense"
            print(f"  [{tool_name}] kept as-is ({result['tokens_before']:,} tokens -- {reason})")

    return pruned, stats


## 6 - Apply Provence to the message history and measure the saving

In [ ]:
current_query = "Create a quote for customer C-1042 for 500 units of SKU-007"

print(f"Current query: {current_query}")
print()
print("=== Provence pruning -- per tool result ===")
pruned_history, prune_stats = prune_history_with_provence(deep_history, current_query)

tokens_before = count_message_tokens(deep_history)
tokens_after  = count_message_tokens(pruned_history)
reduction_pct = 100 * (1 - tokens_after / max(tokens_before, 1))

print()
print("=== Provence pruning results ===")
print(f"  Tool results processed : {prune_stats['tools_processed']}")
print(f"  Tool results pruned    : {prune_stats['tools_pruned']}")
print(f"  Tokens before          : ~{tokens_before:,}")
print(f"  Tokens after           : ~{tokens_after:,}")
print(f"  Reduction              : {reduction_pct:.0f}%")
print()
print("Key insight:")
print("  Provence keeps sentences *relevant to the current query*.")
print("  A Q1 revenue report retains the regional data needed for the current quote")
print("  while discarding the raw row-level data already summarised by the AI turn.")


---
# Part 2 - Packaging Agent Skills from an overloaded system prompt

## 7 - A bloated system prompt: the starting point

Many production Databricks agents have system prompts that have grown organically,
accumulating instructions for capabilities rarely invoked in any given session.


In [ ]:
BLOATED_SYSTEM_PROMPT = """
You are an enterprise data assistant for ACME Corp.
You help users answer questions about customers, products, billing, and revenue.

## Core behaviour
Always respond in the user language. Be concise. Cite the data source for every fact.

## Customer lookup instructions
When looking up a customer, always check both the CRM and billing system.
If the customer_id starts with 'C-', use get_customer_info_v2.
If the account_id starts with 'A-', use lookup_account_details_v2.
Verify the account status before quoting prices.

## Revenue reporting instructions
Always normalise revenue figures to USD. Use Q1=Jan-Mar, Q2=Apr-Jun convention.
When comparing regions, always include EMEA, APAC, and AMER.
Round to 2 decimal places. Include YoY delta if available.
Log every revenue report query to MLflow with run_name='revenue_query'.

## Quote generation instructions (COMPLEX - 40 steps)
Step 1: Verify the customer exists in the CRM.
Step 2: Retrieve the account details from billing.
Step 3: Fetch the product catalog and filter to available SKUs.
Step 4: Calculate the applicable discount using calculate_discount.
Step 5: Compute unit price x quantity x (1 - discount_pct/100).
Step 6: Add tax at the rate for the customer billing region.
... [steps 7-40 continue with invoice formatting, approval rules, PDF generation]
Always cc: finance@acme.com on quotes over $50,000.

## SQL query writing instructions
When writing SQL, always use the {CATALOG}.{SCHEMA} prefix.
Use ANSI SQL. Avoid SELECT *. Add LIMIT 1000 unless the user asks for full results.
Wrap queries in a markdown sql block. Log every SQL query to MLflow.

## Support ticket instructions
Before creating a ticket, always check if a similar open ticket exists.
Use priority: LOW for questions, MEDIUM for bugs, HIGH for outages.
Always include the customer_id, account_id, and product_sku in the ticket.
Escalate to on-call if ticket priority is HIGH and outside business hours.

## PDF report generation instructions (RARE)
Use the pdf skill. Set page size to A4. Use ACME logo from /assets/logo.png.
Include page numbers. Use Helvetica 11pt body, 14pt bold headers.
Tables must have alternating row shading. Export path: /reports/{date}/{report_name}.pdf.

## Data privacy instructions
Never log PII to MLflow. Mask email addresses in logs as user***@domain.
Never include SSN, credit card, or passport numbers in responses.
Apply row-level security: users can only see data their Unity Catalog role permits.
"""

tokens_bloated = len(enc.encode(BLOATED_SYSTEM_PROMPT))
print(f"Bloated system prompt: ~{tokens_bloated:,} tokens")
print("This is loaded into EVERY turn, even when 90% of it is irrelevant.")


## 8 - Analyse each capability block: frequency x size matrix

The packaging decision depends on two axes:
- **Invocation frequency**: how often is this capability needed in a session?
- **Instruction size**: how many tokens does the capability block consume?

High frequency + small = keep in system prompt.
Low frequency + large = package as an on-demand Agent Skill.


In [ ]:
capability_analysis = [
    {"capability": "Core behaviour",                "frequency": "always",   "size": "tiny",       "decision": "KEEP",                    "loading": "always-on"},
    {"capability": "Customer lookup instructions",  "frequency": "frequent", "size": "small",      "decision": "KEEP",                    "loading": "always-on"},
    {"capability": "Revenue reporting instructions","frequency": "moderate", "size": "medium",     "decision": "PACKAGE as Skill",        "loading": "on-demand (trigger: revenue, Q1, region)"},
    {"capability": "Quote generation (40 steps)",  "frequency": "rare",     "size": "very large", "decision": "PACKAGE as Skill",        "loading": "filesystem-resident (fetched on demand)"},
    {"capability": "SQL query writing",            "frequency": "moderate", "size": "small",      "decision": "KEEP",                    "loading": "always-on"},
    {"capability": "Support ticket instructions",  "frequency": "rare",     "size": "medium",     "decision": "PACKAGE as Skill",        "loading": "on-demand (trigger: ticket, bug, support)"},
    {"capability": "PDF report generation",        "frequency": "rare",     "size": "medium",     "decision": "PACKAGE as Skill",        "loading": "filesystem-resident"},
    {"capability": "Data privacy instructions",    "frequency": "always",   "size": "small",      "decision": "KEEP (safety-critical)",  "loading": "always-on"},
]

print(f"{'Capability':<40} {'Freq':>10}  {'Size':<12} {'Decision':<25} {'Loading strategy'}")
print("-" * 120)
for c in capability_analysis:
    print(f"  {c['capability']:<38} {c['frequency']:>10}  {c['size']:<12} {c['decision']:<25} {c['loading']}")


## 9 - Generate the refactored system prompt and Skill stubs

In [ ]:
LEAN_SYSTEM_PROMPT = """
You are an enterprise data assistant for ACME Corp.

## Core behaviour
Always respond in the user language. Be concise. Cite the data source for every fact.

## Customer lookup
If customer_id starts with 'C-', use get_customer_info_v2.
If account_id starts with 'A-', use lookup_account_details_v2.
Verify account status before quoting prices.

## SQL queries
Always use the {CATALOG}.{SCHEMA} prefix. Use ANSI SQL. Avoid SELECT *.
Add LIMIT 1000 unless asked for full results. Wrap queries in sql blocks.

## Data privacy (ALWAYS ENFORCED)
Never log PII to MLflow. Mask emails as user***@domain in logs.
Never include SSN, credit card, or passport numbers in responses.

## Available skills (load on demand)
- revenue_reporting : revenue aggregation, regional breakdowns, YoY comparisons
- quote_generation  : end-to-end quote workflow, pricing, PDF output
- support_tickets   : ticket creation, priority routing, escalation rules
- pdf_reports       : ACME-branded PDF generation with formatting rules
"""

tokens_lean = len(enc.encode(LEAN_SYSTEM_PROMPT))
saving_pct  = 100 * (1 - tokens_lean / tokens_bloated)
print(f"Bloated prompt : ~{tokens_bloated:,} tokens")
print(f"Lean prompt    : ~{tokens_lean:,} tokens")
print(f"Saving         : {saving_pct:.0f}% per turn")


In [ ]:
# SKILL.md stubs -- in production these live in a /skills directory
# accessible to the agent (Databricks Workspace Files, Repos, or local filesystem)

skills = {
    "skills/revenue_reporting/SKILL.md": """\
---
name: revenue_reporting
description: "Revenue aggregation, regional breakdowns, and YoY comparisons for ACME Corp.
Load when the user asks about revenue, financial performance, quarterly results, or regional sales."
---

# Revenue Reporting

## Conventions
- Normalise all figures to USD
- Q1 = Jan-Mar, Q2 = Apr-Jun, Q3 = Jul-Sep, Q4 = Oct-Dec
- Always include EMEA, APAC, and AMER in regional comparisons
- Round to 2 decimal places; include YoY delta if available
- Log every revenue query to MLflow with run_name='revenue_query'

## Tool to use
run_revenue_report(period: str)  -- pass 'Q1', 'Q2', 'Q3', or 'Q4'
""",

    "skills/quote_generation/SKILL.md": """\
---
name: quote_generation
description: "End-to-end quote generation: customer verification, pricing, discount,
tax, PDF output. Load when the user requests a quote, proposal, or pricing estimate."
---

# Quote Generation

## Steps
1. Verify customer -> get_customer_info_v2
2. Retrieve account plan -> lookup_account_details_v2
3. Fetch available SKUs -> get_product_catalog
4. Compute discount -> calculate_discount(account_id, volume)
5. Calculate: unit_price x qty x (1 - discount_pct/100) + tax(region)
6. CC finance@acme.com on any quote > $50,000
7. For PDF output -> see pdf_reports skill
""",

    "skills/support_tickets/SKILL.md": """\
---
name: support_tickets
description: "Support ticket creation, priority routing, and escalation rules.
Load when the user reports a problem, bug, outage, or requests human follow-up."
---

# Support Ticket Workflow
1. Check for duplicate open tickets first
2. Assign priority: LOW (questions), MEDIUM (bugs), HIGH (outages)
3. Include customer_id, account_id, product_sku in every ticket
4. Escalate HIGH priority tickets outside business hours to on-call
5. Tool: send_support_ticket(customer_id, priority, description)
""",
}

import os

for path, content in skills.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)
    yaml_tokens = len(enc.encode(content.split("---")[1])) if "---" in content else 0
    full_tokens  = len(enc.encode(content))
    print(f"  {path}")
    print(f"    Discovery cost (YAML header only) : ~{yaml_tokens:>4} tokens")
    print(f"    Full body cost (when activated)   : ~{full_tokens:>4} tokens")
    print()


## 10 - Loading strategy decision tree

In [ ]:
print("""
=== Agent Skill Loading Strategy Decision Tree ===

Is this capability needed in every session?
+-- YES -> Keep in system prompt (always-on)
|         Examples: core behaviour, safety rules
|
+-- NO  -> Is it triggered by recognisable keywords?
          +-- YES -> On-demand Skill (SKILL.md with YAML header)
          |         Agent loads name + description only (~80 tokens)
          |         Full body loaded only when trigger matches
          |         Examples: revenue_reporting, support_tickets
          |
          +-- NO  -> Is it very large (> 500 tokens)?
                    +-- YES -> Filesystem-resident Skill
                    |         SKILL.md references additional files
                    |         Files fetched from disk only during execution
                    |         Examples: quote_generation, pdf_reports
                    |
                    +-- NO  -> On-demand Skill (same as above, smaller body)

Databricks integration:
  - Store /skills directory in Databricks Workspace Files or Repos
  - SKILL.md manifests version-controlled alongside agent code
  - Combine with MCP Catalog for governed skill discovery
""")


## 11 - End-to-end token budget summary

In [ ]:
skill_discovery_tokens = sum(
    len(enc.encode(c.split("---")[1]))
    for c in skills.values()
    if "---" in c
)

tokens_turn_before = tokens_bloated + count_message_tokens(deep_history)
tokens_turn_after  = tokens_lean + count_message_tokens(pruned_history) + skill_discovery_tokens

print("=== Full Turn Token Budget ===")
print()
print("  BEFORE optimisation:")
print(f"    Bloated system prompt : {tokens_bloated:>7,} tokens")
print(f"    Deep history (raw)    : {count_message_tokens(deep_history):>7,} tokens")
print(f"    Total                 : {tokens_turn_before:>7,} tokens")
print()
print("  AFTER optimisation:")
print(f"    Lean system prompt    : {tokens_lean:>7,} tokens")
print(f"    Deep history (pruned) : {count_message_tokens(pruned_history):>7,} tokens")
print(f"    Skill discovery       : {skill_discovery_tokens:>7,} tokens")
print(f"    Total                 : {tokens_turn_after:>7,} tokens")
print()
total_saving = 100 * (1 - tokens_turn_after / tokens_turn_before)
print(f"  Net reduction : {total_saving:.0f}% per turn")


---
## 12 - Key takeaways

### Pruning tool outputs with Provence

| Situation | Provence behaviour |
|-----------|-------------------|
| Large result, partially relevant | Keeps only relevant sentences -- better than full eviction |
| Small result (< threshold) | Skipped -- no overhead for tiny payloads |
| Low reranking score | Signal that the whole result may be irrelevant to the current task |
| Dense, all-relevant result | Returned unchanged -- Provence does not force pruning |

**Why Provence over heuristics:**
- Heuristics evict entire ToolMessages based on size/age -- blunt and lossy
- Provence compresses semantically: keeps the sentences that answer the current query
- A 300-row product catalog can be reduced to the 2 sentences needed for a specific quote

### Packaging Agent Skills

| Frequency x Size | Loading strategy |
|------------------|------------------|
| Always + small | Keep in system prompt |
| Moderate / rare + medium | On-demand Skill (YAML trigger) |
| Rare + very large | Filesystem-resident Skill (fetched during execution) |
| Safety-critical (any size) | Always keep in system prompt |

**Combined impact:** Provence pruning + lean system prompt + skill discovery
can reduce per-turn token consumption by 40-70% while preserving the signal needed for the current step.
